<a href="https://colab.research.google.com/github/hernans7/mis433/blob/main/Copy_of_data_summarization_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise: Data Summarization

In [1]:
import pandas as pd
import numpy as np

In [3]:
# Install the package 'nycflights13' before you can run this
!pip install nycflights13
from nycflights13 import flights
flights.head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 79.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for nycflights13: filename=nycflights13-0.0.3-py3-none-any.whl size=8732722 sha256=4b40559738fba3b7775c5e0afbad2796d421d84917136e17ceead03b9d38e6d7
  Stored in directory: /root/.cache/pip/wheels/58/ee/20/7838832df5510701439239ec10c2d8927de32aac2729ce2085
Successfully built nycflights13


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


## DataFrame with columns

- year,month,day
        Date of departure    
- dep_time,arr_time
        Actual departure and arrival times (format HHMM or HMM), local tz.
- sched_dep_time,sched_arr_time
        Scheduled departure and arrival times (format HHMM or HMM), local tz.    
- dep_delay,arr_delay
        Departure and arrival delays, in minutes. Negative times represent early departures/arrivals.
- hour,minute
        Time of scheduled departure broken into hour and minutes.
- carrier
        Two letter carrier abbreviation. See airlines() to get name
- tailnum
        Plane tail number
- flight
        Flight number
- origin,dest
        Origin and destination. See airports() for additional metadata.
- air_time
        Amount of time spent in the air, in minutes
- distance
        Distance between airports, in miles
- time_hour
        Scheduled date and hour of the flight as a date. Along with origin, can be used to join flights data to weather data.

In [4]:
flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336776 entries, 0 to 336775
Data columns (total 19 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   year            336776 non-null  int64  
 1   month           336776 non-null  int64  
 2   day             336776 non-null  int64  
 3   dep_time        328521 non-null  float64
 4   sched_dep_time  336776 non-null  int64  
 5   dep_delay       328521 non-null  float64
 6   arr_time        328063 non-null  float64
 7   sched_arr_time  336776 non-null  int64  
 8   arr_delay       327346 non-null  float64
 9   carrier         336776 non-null  object 
 10  flight          336776 non-null  int64  
 11  tailnum         334264 non-null  object 
 12  origin          336776 non-null  object 
 13  dest            336776 non-null  object 
 14  air_time        327346 non-null  float64
 15  distance        336776 non-null  int64  
 16  hour            336776 non-null  int64  
 17  minute    

## Exercises

Write Python scripts to answer the following questions.

In [5]:
# How many distinct airplanes are there in this dataset?
distinct_airplanes = flights['tailnum'].nunique()
print(f"There are {distinct_airplanes} distinct airplanes in the dataset.")

There are 4043 distinct airplanes in the dataset.


In [6]:
# Which hours of the day are the busiest?
# List the top 5 hours with the most flights.

busiest_hours = flights['hour'].value_counts().nlargest(5)
print("Top 5 busiest hours of the day:")
print(busiest_hours)

Top 5 busiest hours of the day:
hour
8     27242
6     25951
17    24426
15    23888
16    23002
Name: count, dtype: int64


In [7]:
# Find all flights (identified by 'carrier' and 'flight')
# that always depart at least 60 minutes late.


not_always_late_flights = flights[(flights['dep_delay'] < 60) | (flights['dep_delay'].isnull())]


not_always_late_combinations = not_always_late_flights[['carrier', 'flight']].drop_duplicates()


always_late_flights_df = flights.merge(
    not_always_late_combinations,
    on=['carrier', 'flight'],
    how='left',
    indicator=True
)

# Filter for flights where the indicator is 'left_only', meaning they are not in the 'not_always_late_combinations' set
always_60min_late = always_late_flights_df[always_late_flights_df['_merge'] == 'left_only']

# Get the unique carrier and flight combinations that always depart at least 60 minutes late
always_60min_late_combinations = always_60min_late[['carrier', 'flight']].drop_duplicates()

print("Flights (carrier, flight) that always depart at least 60 minutes late:")
display(always_60min_late_combinations)


Flights (carrier, flight) that always depart at least 60 minutes late:


,carrier,flight
1407,EV,4617
1499,WN,2521
1569,WN,2529
1583,EV,4151
25525,OO,8500
26278,US,1226
38024,9E,3283
57452,9E,3297
78484,YV,3799
78964,DL,809


In [8]:
# How many flights always depart at least 60 minutes late?
num_always_60min_late_flights = len(always_60min_late_combinations)
print(f"There are {num_always_60min_late_flights} flights that always depart at least 60 minutes late.")

There are 52 flights that always depart at least 60 minutes late.


In [9]:
# Find all United Airlines (UA) flights (identified by 'carrier' and 'flight')
# that depart (at least) 60 minutes late.

ua_late_flights = flights[
    (flights['carrier'] == 'UA') &
    (flights['dep_delay'] >= 60)
]

ua_late_flight_combinations = ua_late_flights[['carrier', 'flight']].drop_duplicates()

print("United Airlines (UA) flights that depart at least 60 minutes late:")
display(ua_late_flight_combinations)

United Airlines (UA) flights that depart at least 60 minutes late:


,carrier,flight
218,UA,856
268,UA,1086
526,UA,465
1032,UA,651
1310,UA,468
...,...,...
316596,UA,1496
320664,UA,1421
322130,UA,230
325949,UA,382


In [10]:
# How many United Airlines (UA) flights depart (at least) 60 minutes late?
num_ua_late_flights = len(ua_late_flight_combinations)
print(f"There are {num_ua_late_flights} United Airlines (UA) flights that depart at least 60 minutes late.")

There are 854 United Airlines (UA) flights that depart at least 60 minutes late.


In [11]:
# Which flights that always arrived on time in December?

# December flights
december_flights = flights[flights['month'] == 12]

# carrier and flight combinations in December
all_dec_flight_combinations = december_flights[['carrier', 'flight']].drop_duplicates()

# December Delays
late_dec_flights = december_flights[december_flights['arr_delay'] > 0]


late_dec_flight_combinations = late_dec_flights[['carrier', 'flight']].drop_duplicates()

# Find flights that are in all_dec_flight_combinations but NOT in late_dec_flight_combinations
always_on_time_december_flights = all_dec_flight_combinations[~all_dec_flight_combinations.isin(late_dec_flight_combinations).all(axis=1)]

print("Flights (carrier, flight) that always arrived on time in December:")
display(always_on_time_december_flights)

Flights (carrier, flight) that always arrived on time in December:


,carrier,flight
83163,US,1895
83164,UA,1487
83165,AA,2243
83166,B6,939
83167,EV,3819
...,...,...
111090,WN,2441
111117,WN,688
111139,EV,5012
111156,EV,5003


In [12]:
# How many flights always arrived on time in December?
num_always_on_time_december_flights = len(always_on_time_december_flights)
print(f"There are {num_always_on_time_december_flights} flights that always arrived on time in December.")

There are 1325 flights that always arrived on time in December.


In [13]:
# Create a dataframe 'cancelled' for all cancelled flights.
# Assumption: flights that are not cancelled should have values for dep_time.
cancelled = flights[flights['dep_time'].isnull()]
display(cancelled.head())

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
838,2013,1,1,NaN,1630,NaN,NaN,1815,NaN,EV,4308,N18120,EWR,RDU,NaN,416,16,30,2013-01-01T21:00:00Z
839,2013,1,1,NaN,1935,NaN,NaN,2240,NaN,AA,791,N3EHAA,LGA,DFW,NaN,1389,19,35,2013-01-02T00:00:00Z
840,2013,1,1,NaN,1500,NaN,NaN,1825,NaN,AA,1925,N3EVAA,LGA,MIA,NaN,1096,15,0,2013-01-01T20:00:00Z
841,2013,1,1,NaN,600,NaN,NaN,901,NaN,B6,125,N618JB,JFK,FLL,NaN,1069,6,0,2013-01-01T11:00:00Z
1777,2013,1,2,NaN,1540,NaN,NaN,1747,NaN,EV,4352,N10575,EWR,CVG,NaN,569,15,40,2013-01-02T20:00:00Z


In [14]:
# Look at the number of cancelled flights per day.
# Find the top 10 days with the most cancelled flights.

cancelled_per_day = cancelled.groupby(['year', 'month', 'day']).size().reset_index(name='num_cancelled')
top_10_cancelled_days = cancelled_per_day.sort_values(by='num_cancelled', ascending=False).head(10)

print("Top 10 days with the most cancelled flights:")
display(top_10_cancelled_days)

Top 10 days with the most cancelled flights:


,year,month,day,num_cancelled
38,2013,2,8,472
39,2013,2,9,393
140,2013,5,23,221
336,2013,12,10,204
251,2013,9,12,192
64,2013,3,6,180
66,2013,3,8,180
331,2013,12,5,158
340,2013,12,14,125
175,2013,6,28,123


In [17]:
# Which carrier has the worst average arrival delay?

print(f"The carrier with the worst average arrival delay is {worst_carrier} with an average delay of {worst_average_delay:.2f} minutes.")

The carrier with the worst average arrival delay is F9 with an average delay of 21.92 minutes.


In [19]:
# Which plane (tailnum) has the worst on-time record?

print(f"The plane with the worst on-time record (highest average arrival delay) is {worst_plane_tailnum} with an average delay of {worst_plane_average_delay:.2f} minutes.")

The plane with the worst on-time record (highest average arrival delay) is N844MH with an average delay of 320.00 minutes.


In [25]:
# The best time (hour) to fly to avoid delays as much as possibl
average_delay_per_hour = flights.groupby('hour')['dep_delay'].mean()
best_hour = average_delay_per_hour.idxmin()
min_average_delay = average_delay_per_hour.min()

print(f"The best time (hour) to fly to avoid delays as much as possible is hour {int(best_hour)} with an average delay of {min_average_delay:.2f} minutes.")

The best time (hour) to fly to avoid delays as much as possible is hour 5 with an average delay of 0.69 minutes.


In [22]:
# For each destination, compute the total minutes of arrival delay.
total_arrival_delay_per_destination = flights.groupby('dest')['arr_delay'].sum().reset_index()
total_arrival_delay_per_destination.rename(columns={'arr_delay': 'total_arr_delay_minutes'}, inplace=True)

print("Total minutes of arrival delay per destination:")
display(total_arrival_delay_per_destination.sort_values(by='total_arr_delay_minutes', ascending=False).head())

Total minutes of arrival delay per destination:


,dest,total_arr_delay_minutes
4,ATL,190260.0
23,CLT,100645.0
69,ORD,97352.0
35,FLL,96153.0
28,DCA,82609.0


In [23]:
# Find the worst days in terms of average arrival delay greater than 1 hour.
average_daily_arrival_delay = flights.groupby(['year', 'month', 'day'])['arr_delay'].mean().reset_index()
worst_days = average_daily_arrival_delay[average_daily_arrival_delay['arr_delay'] > 60]
worst_days_sorted = worst_days.sort_values(by='arr_delay', ascending=False)

print("Worst days in terms of average arrival delay greater than 1 hour:")
display(worst_days_sorted.head())

Worst days in terms of average arrival delay greater than 1 hour:


,year,month,day,arr_delay
66,2013,3,8,85.862155
163,2013,6,13,63.753689
202,2013,7,22,62.763403
142,2013,5,23,61.970899


In [24]:
# Find the number of carriers between each pair of origin and destination.
carrier_count_per_route = flights.groupby(['origin', 'dest'])['carrier'].nunique().reset_index(name='num_carriers')

print("Number of carriers between each pair of origin and destination:")
display(carrier_count_per_route.head())

Number of carriers between each pair of origin and destination:


,origin,dest,num_carriers
0,EWR,ALB,1
1,EWR,ANC,1
2,EWR,ATL,4
3,EWR,AUS,2
4,EWR,AVL,1
